# 00. v0 → v1 마이그레이션 가이드

LangChain/LangGraph v0에서 v1으로 전환할 때 알아야 할 브레이킹 체인지와 코드 매핑을 다룹니다.

## 학습 목표

- v1 패키지 구조 변경과 import 경로를 이해한다
- `create_react_agent` → `create_agent` 마이그레이션을 수행한다
- 미들웨어 기반 동적 프롬프트, 상태 관리, 컨텍스트 주입을 적용한다
- 표준 콘텐츠 블록과 구조화된 출력 전략을 활용한다

## 0.1 패키지 구조 변경

v1에서 `langchain` 네임스페이스가 에이전트 구축에 필수적인 5개 핵심 모듈로 대폭 축소되었습니다:

| v1 모듈 | 역할 | 주요 API |
|---|---|---|
| `langchain.agents` | 에이전트 생성 및 상태 관리 | `create_agent`, `AgentState` |
| `langchain.messages` | 메시지 타입과 콘텐츠 블록 | `HumanMessage`, `AIMessage`, `content_blocks` |
| `langchain.tools` | 도구 정의 | `@tool` 데코레이터, `BaseTool` |
| `langchain.chat_models` | 모델 초기화 | `init_chat_model` |
| `langchain.embeddings` | 임베딩 유틸리티 | 임베딩 모델 래퍼 |

### 레거시 코드 마이그레이션 — `langchain-classic`

기존에 `langchain` 패키지에서 사용하던 Chains, Retrievers, Hub, 인덱싱 API 등은 모두 `langchain-classic`이라는 별도 패키지로 분리되었습니다. 기존 코드를 유지해야 하는 경우 `pip install langchain-classic`으로 설치 후 import 경로를 변경합니다:

```python
# v0 — 기존 방식
from langchain.chains import LLMChain
from langchain.retrievers import MultiQueryRetriever
from langchain import hub

# v1 — langchain-classic으로 이전
from langchain_classic.chains import LLMChain
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_classic import hub
```

이 분리를 통해 v1의 `langchain` 패키지는 에이전트 빌딩에만 집중하는 경량 구조가 되었으며, 레거시 기능은 독립적으로 유지보수됩니다.

## 0.2 에이전트 생성 API 변경

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent

model = ChatOpenAI(model="gpt-4.1")

@tool
def add(a: int, b: int) -> int:
    """두 수를 더합니다."""
    return a + b

# v0: from langgraph.prebuilt import create_react_agent
# v1: from langchain.agents import create_agent
agent = create_agent(
    model=model,
    tools=[add],
    system_prompt="당신은 수학 어시스턴트입니다.",  # v0: prompt=
)
print("\u2713 v1 에이전트 생성 완료")

✓ v1 에이전트 생성 완료


In [2]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — https://lf.ddok.ai


### 주요 변경점

| v0 | v1 | 비고 |
|---|---|---|
| `from langgraph.prebuilt import create_react_agent` | `from langchain.agents import create_agent` | import 경로 + 함수명 |
| `prompt=` | `system_prompt=` | 파라미터명 |
| `ToolNode` 지원 | 미지원 | 함수/BaseTool/dict만 |
| `pre_hooks`, `post_hooks` | `middleware=[]` | 통합 미들웨어 |
| Pydantic/dataclass 상태 | `TypedDict` only | 상태 스키마 |

## 0.3 상태 스키마 — TypedDict only

v1에서 커스텀 상태는 반드시 `TypedDict` 기반 `AgentState`를 상속해야 합니다.

In [3]:
from langchain.agents import AgentState, create_agent

class CustomState(AgentState):
    user_id: str

agent = create_agent(
    model=model,
    tools=[add],
    state_schema=CustomState,
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "3+4는 얼마인가요?"}],
    "user_id": "user_123",
}, config=lf_config)
print(result["messages"][-1].content)

3 + 4는 7입니다.


## 0.4 런타임 컨텍스트 주입 (신규)

v1에서는 `context_schema`와 `context` 파라미터를 통해 **불변 런타임 데이터**를 에이전트에 전달할 수 있습니다. 이는 사용자 ID, 역할, 세션 정보 등 요청마다 달라지지만 실행 중에는 변하지 않는 데이터를 에이전트와 도구에 안전하게 전달하는 패턴입니다.

**작동 방식:**
1. `@dataclass`로 컨텍스트 스키마를 정의합니다.
2. `create_agent(context_schema=...)`로 스키마를 등록합니다.
3. `agent.invoke(..., context=ContextInstance(...))`로 런타임에 값을 전달합니다.
4. 도구에서는 `ToolRuntime[ContextType]` 파라미터로 컨텍스트에 접근합니다.

컨텍스트는 에이전트 상태(`AgentState`)와 달리 도구 호출 간에 **변경되지 않는 읽기 전용 데이터**입니다. 상태는 에이전트 루프 중 업데이트되지만, 컨텍스트는 `invoke` 호출 시 고정됩니다.

In [4]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@dataclass
class UserContext:
    user_id: str
    role: str

@tool
def get_role(runtime: ToolRuntime[UserContext]) -> str:
    """현재 사용자의 역할을 조회합니다."""
    return f"사용자 {runtime.context.user_id}의 역할: {runtime.context.role}"

agent = create_agent(
    model=model,
    tools=[get_role],
    context_schema=UserContext,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "제 역할이 뭔가요?"}]},
    context=UserContext(user_id="u1", role="admin"),
    config=lf_config,
)
print(result["messages"][-1].content)

당신의 역할은 "admin"입니다. 즉, 관리자 권한을 가지고 있습니다. 추가로 궁금한 점이 있으시면 말씀해 주세요.


## 0.5 동적 프롬프트 — 미들웨어 방식 (신규)

v0의 정적 프롬프트 대신, `@dynamic_prompt`로 런타임 컨텍스트에 따라 프롬프트를 동적 생성합니다.

In [5]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def role_based_prompt(request: ModelRequest) -> str:
    role = request.runtime.context.role
    if role == "admin":
        return "당신은 전체 접근 권한을 가진 관리자 어시스턴트입니다."
    return "당신은 읽기 전용 어시스턴트입니다."

agent = create_agent(
    model=model,
    tools=[add],
    context_schema=UserContext,
    middleware=[role_based_prompt],
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요!"}]},
    context=UserContext(user_id="u1", role="admin"),
    config=lf_config,
)
print(result["messages"][-1].content)

안녕하세요! 무엇을 도와드릴까요? 😊


## 0.6 도구 에러 처리 — `@wrap_tool_call` (신규)

v0의 `handle_tool_errors` 대신, v1은 미들웨어로 도구 에러를 처리합니다.

In [6]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@wrap_tool_call
def handle_errors(request, handler):
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"도구 오류: {e}",
            tool_call_id=request.tool_call["id"],
        )

agent = create_agent(
    model=model,
    tools=[add],
    middleware=[handle_errors],
)
print("\u2713 에러 핸들링 미들웨어 적용")

✓ 에러 핸들링 미들웨어 적용


## 0.7 표준 콘텐츠 블록 & 구조화된 출력 (신규)

v1에서 메시지는 프로바이더 무관한 `content_blocks`를 지원합니다.
구조화된 출력은 `ToolStrategy`(도구 호출 기반)와 `ProviderStrategy`(네이티브) 두 가지로 분리되었습니다.

In [7]:
# 표준 콘텐츠 블록

response = model.invoke(
    "인사해 주세요.",
    config=lf_config
)

for block in response.content_blocks:
    print(f"  [{block['type']}] {block.get('text', '')[:100]}")

# .text 프로퍼티 (v0: .text() 메서드 → v1: .text 프로퍼티)

print(f"\n.text: {response.text[:100]}")

  [text] 안녕하세요! 저는 여러분을 도와드릴 준비가 되어 있는 AI 챗봇입니다. 궁금하신 점이 있거나 도움이 필요하시면 언제든 말씀해 주세요. 😊

.text: 안녕하세요! 저는 여러분을 도와드릴 준비가 되어 있는 AI 챗봇입니다. 궁금하신 점이 있거나 도움이 필요하시면 언제든 말씀해 주세요. 😊


## 0.8 스트리밍 변경

| 항목 | v0 | v1 |
|---|---|---|
| 에이전트 노드명 | `"agent"` | `"model"` |
| `.text` | 메서드 `.text()` | 프로퍼티 `.text` |

In [8]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "5+3은 얼마인가요?"}]},
    context=UserContext(user_id="u1", role="admin"),
    stream_mode="updates",
    config=lf_config,
):
    for node_name, update in chunk.items():
        # v1: node_name은 "model" (v0에서는 "agent")
        if "messages" in update:
            last = update["messages"][-1]
            if hasattr(last, "content") and last.content:
                print(f"[{node_name}] {last.content[:200]}")

[tools] 8


[model] 5 + 3의 결과는 8입니다.


## 0.9 기타 브레이킹 체인지

| 변경 | 설명 |
|---|---|
| **Python 3.10+** | 모든 LangChain 패키지가 Python 3.10 이상을 요구합니다. 3.9 이하는 미지원됩니다. |
| **반환 타입** | 채팅 모델의 반환 타입이 `BaseMessage`에서 `AIMessage`로 고정되었습니다. |
| **OpenAI Responses API** | 메시지 콘텐츠가 기본적으로 표준 블록 형식입니다. `output_version="v0"`으로 이전 동작을 복구할 수 있습니다. |
| **Anthropic max_tokens** | 기본값이 1024에서 모델별 자동 설정으로 변경되었습니다. |
| **AIMessage.example** | `example` 파라미터가 제거되었습니다. `additional_kwargs`를 사용하세요. |
| **AIMessageChunk** | `chunk_position` 속성이 추가되었습니다 (마지막 청크에 `'last'` 값). |
| **`.text` 프로퍼티** | `.text()` 메서드가 `.text` 프로퍼티로 변경되었습니다. |
| **파일 인코딩** | 파일이 기본적으로 UTF-8 인코딩으로 열립니다. |

## 요약 — 마이그레이션 체크리스트

- [ ] Python 3.10+ 확인
- [ ] `create_react_agent` → `create_agent` 변경
- [ ] `prompt=` → `system_prompt=` 변경
- [ ] 상태 스키마를 `TypedDict` 기반 `AgentState`로 전환
- [ ] `pre_hooks`/`post_hooks` → `middleware=[]`
- [ ] `ToolNode` → 함수/BaseTool로 교체
- [ ] `.text()` → `.text` 프로퍼티
- [ ] 스트리밍 노드명 `"agent"` → `"model"` 확인
- [ ] 레거시 import를 `langchain-classic`으로 이전

### 다음 단계
→ **[01_middleware.ipynb](./01_middleware.ipynb)**: v1 최대 신기능 — 미들웨어 시스템 심화